# Data Visualization with Matplotlib

Visualization turns numbers into pictures — making patterns, trends, and outliers visible at a glance. This notebook teaches you how to create common chart types using matplotlib, Python's foundational plotting library.

## When to Use Each Chart Type
- **Bar charts**: Compare quantities across categories (e.g., items per category)
- **Pie charts**: Show parts of a whole (e.g., percentage breakdown)
- **Horizontal bar charts**: Compare values with long labels (e.g., recipe names)
- **Timeline/line charts**: Show trends over time (e.g., purchases by date)

## Matplotlib Key Concepts
- **Figure**: The overall image/window (like a canvas)
- **Axes**: A single chart within the figure (the actual plot area)
- `plt.subplots()`: Creates both at once
- `plt.savefig()`: Saves to a file (PNG, PDF, etc.)
- `plt.show()`: Displays interactively in Jupyter

In [ ]:
import sys
import json
from pathlib import Path

# Add the project root to Python's module search path
project_root = str(Path.cwd().parent) if Path.cwd().name == "notebooks" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from src.database import get_engine

# Enable inline display in Jupyter notebooks
%matplotlib inline

# Set a clean style for all charts
plt.style.use("seaborn-v0_8-whitegrid")

# Load data from the database
engine = get_engine()
recipes_df = pd.read_sql("SELECT * FROM recipe", engine)
receipts_df = pd.read_sql("SELECT * FROM receipt", engine)
inventory_df = pd.read_sql("SELECT * FROM activeinventory", engine)
match_summary_df = pd.read_sql("SELECT * FROM recipematchsummary", engine)
match_detail_df = pd.read_sql("SELECT * FROM recipeingredientmatch", engine)

print("Data loaded successfully!")
print(f"  {len(recipes_df)} recipes, {len(receipts_df)} receipts, "
      f"{len(inventory_df)} inventory items")

## 1. Bar Charts — Comparing Quantities

Bar charts are the workhorse of data visualization. Use them when you want to compare a numeric value across categories.

In [ ]:
# CHART: Top 15 Most Common Ingredients Across All Recipes
# This shows which ingredients appear in the most recipes

# Step 1: Parse JSON ingredients and extract names
recipes_df["parsed_ingredients"] = recipes_df["ingredients"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

ingredient_names = recipes_df["parsed_ingredients"].apply(
    lambda ings: [ing["name"] for ing in ings if isinstance(ing, dict)]
)

# Step 2: Explode lists into individual rows, then count
top_ingredients = ingredient_names.explode().value_counts().head(15)

# Step 3: Create the bar chart
fig, ax = plt.subplots(figsize=(10, 6))  # Create figure (10" wide, 6" tall)

# ax.bar() draws vertical bars
bars = ax.bar(
    range(len(top_ingredients)),      # x-positions (0, 1, 2, ...)
    top_ingredients.values,           # bar heights (counts)
    color="#3498db",                  # bar color (blue)
    edgecolor="white",               # white border between bars
    width=0.7,                       # bar width (0 to 1, where 1 = no gap)
)

# Customize x-axis labels (rotate for readability)
ax.set_xticks(range(len(top_ingredients)))
ax.set_xticklabels(top_ingredients.index, rotation=45, ha="right")

# Add count labels on top of each bar
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 0.1,
            str(int(height)), ha="center", va="bottom", fontsize=9)

# Label the chart
ax.set_xlabel("Ingredient")          # x-axis label
ax.set_ylabel("Number of Recipes")   # y-axis label
ax.set_title("Top 15 Most Common Recipe Ingredients")  # chart title

plt.tight_layout()  # Adjust spacing to prevent label cutoff
plt.show()          # Display in the notebook

In [ ]:
# CHART: Inventory Items by Category
# Shows the distribution of your kitchen inventory across food categories

# Count items per category (only active, non-expired items)
active_inv = inventory_df[inventory_df["is_expired"] == False]
cat_counts = active_inv["category"].value_counts()

fig, ax = plt.subplots(figsize=(9, 5))

# Use a colormap for variety — cm.Set2 gives distinct, pleasant colors
colors = plt.cm.Set2(range(len(cat_counts)))

ax.bar(cat_counts.index, cat_counts.values, color=colors, edgecolor="white")

ax.set_xlabel("Category")
ax.set_ylabel("Number of Items")
ax.set_title("Active Inventory by Category")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# CHART: Spending by Store
# Compare how much you spend at each store

if "total_price" in receipts_df.columns and receipts_df["total_price"].notna().any():
    store_totals = receipts_df.groupby("store_name")["total_price"].sum().sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(store_totals.index, store_totals.values, color="#2ecc71", edgecolor="white")
    
    # Add dollar labels on top
    for bar, amount in zip(bars, store_totals.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"${amount:.2f}", ha="center", fontsize=10)
    
    ax.set_ylabel("Total Spent ($)")
    ax.set_title("Total Spending by Store")
    plt.tight_layout()
    plt.show()
else:
    print("No price data available to chart spending by store.")

## 2. Pie Charts — Parts of a Whole

Pie charts show how a total breaks down into categories. They work best with 3-7 categories. More than that becomes hard to read — switch to a bar chart instead.

In [ ]:
# CHART: Recipe Weather Distribution
# Shows how your recipes are split across weather categories

fig, axes = plt.subplots(1, 2, figsize=(12, 5))  # 1 row, 2 charts side by side

# Left pie: weather temperature
temp_counts = recipes_df["weather_temp"].fillna("unset").value_counts()
axes[0].pie(
    temp_counts.values,
    labels=temp_counts.index,
    autopct="%1.0f%%",        # Show percentages with 0 decimal places
    startangle=90,            # Start first slice at 12 o'clock
    colors=["#e74c3c", "#3498db", "#95a5a6"],  # red for warm, blue for cold
)
axes[0].set_title("By Temperature")

# Right pie: weather condition
cond_counts = recipes_df["weather_condition"].fillna("unset").value_counts()
axes[1].pie(
    cond_counts.values,
    labels=cond_counts.index,
    autopct="%1.0f%%",
    startangle=90,
    colors=plt.cm.Pastel1.colors[:len(cond_counts)],
)
axes[1].set_title("By Condition")

plt.suptitle("Recipe Weather Distribution", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# CHART: Where Does Your Inventory Come From?
# Shows the split between receipt-sourced and pantry-sourced items

source_counts = inventory_df["source"].value_counts()

fig, ax = plt.subplots(figsize=(6, 6))

# explode slightly separates slices for emphasis
explode = [0.03] * len(source_counts)

ax.pie(
    source_counts.values,
    labels=source_counts.index,
    autopct="%1.0f%%",
    startangle=90,
    explode=explode,
    colors=["#3498db", "#e67e22"],
    shadow=True,              # Adds a subtle drop shadow
)
ax.set_title("Inventory by Data Source")
plt.tight_layout()
plt.show()

## 3. Horizontal Bar Charts — Rankings and Lists

Horizontal bar charts are ideal when labels are long text (like recipe names or ingredient lists). They're also great for "ranked" data — sorted lists where position matters.

In [ ]:
# CHART: Most Commonly Missing Ingredients (Shopping List Visualization)
# These are ingredients needed by recipes but not in your inventory

missing = match_detail_df[match_detail_df["is_available"] == False]

if not missing.empty:
    missing_counts = missing["ingredient_name"].value_counts().head(15)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # barh() draws HORIZONTAL bars — y-axis has names, x-axis has values
    # We reverse so the most-needed ingredient is at the top
    ax.barh(
        range(len(missing_counts)),
        missing_counts.values[::-1],
        color="#e74c3c",
        edgecolor="white",
    )
    
    # Set y-axis labels to ingredient names (reversed to match bars)
    ax.set_yticks(range(len(missing_counts)))
    ax.set_yticklabels(missing_counts.index[::-1])
    
    ax.set_xlabel("Number of Recipes Needing This")
    ax.set_title("Top Missing Ingredients — Your Shopping Priority List")
    
    plt.tight_layout()
    plt.show()
else:
    print("No missing ingredients found — you can make everything!")

In [ ]:
# CHART: Recipe Makeability — How Close Are You?
# Shows each recipe with available vs. missing ingredient counts

if not match_summary_df.empty:
    # Sort by available ingredients percentage
    chart_data = match_summary_df.copy()
    chart_data["pct_available"] = (
        chart_data["available_ingredients"] / chart_data["total_ingredients"] * 100
    )
    chart_data = chart_data.sort_values("pct_available", ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, max(4, len(chart_data) * 0.35)))
    
    y_pos = range(len(chart_data))
    
    # Stacked horizontal bars: green for available, red for missing
    ax.barh(y_pos, chart_data["available_ingredients"], color="#27ae60",
            label="Available", edgecolor="white")
    ax.barh(y_pos, chart_data["missing_ingredients"],
            left=chart_data["available_ingredients"],  # Stack on top
            color="#e74c3c", label="Missing", edgecolor="white")
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(chart_data["recipe_name"])
    ax.set_xlabel("Number of Ingredients")
    ax.set_title("Recipe Ingredient Availability")
    ax.legend(loc="lower right")
    
    plt.tight_layout()
    plt.show()

## 4. Timeline Charts — Data Over Time

Timeline charts show how values change over time. They're essential for spotting trends, seasonality, and patterns in your purchasing or inventory data.

In [ ]:
# CHART: Expiration Timeline — What's Expiring When?
# Shows how many items expire each week over the coming month

from datetime import date, timedelta

active_inv = inventory_df[
    (inventory_df["is_expired"] == False) & 
    (inventory_df["expiration_date"].notna())
].copy()

if not active_inv.empty:
    # Convert expiration_date to datetime for matplotlib
    active_inv["exp_date"] = pd.to_datetime(active_inv["expiration_date"])
    
    today = pd.Timestamp(date.today())
    one_month = today + pd.Timedelta(days=30)
    
    # Filter to items expiring in the next 30 days
    upcoming = active_inv[
        (active_inv["exp_date"] >= today) & 
        (active_inv["exp_date"] <= one_month)
    ]
    
    if not upcoming.empty:
        fig, ax = plt.subplots(figsize=(10, 5))
        
        # Count items per expiration date
        exp_counts = upcoming.groupby("exp_date").size().reset_index(name="count")
        
        # Color bars by urgency
        colors = []
        for d in exp_counts["exp_date"]:
            days_away = (d - today).days
            if days_away <= 3:
                colors.append("#e74c3c")   # Red: urgent
            elif days_away <= 7:
                colors.append("#f39c12")   # Orange: soon
            else:
                colors.append("#27ae60")   # Green: has time
        
        ax.bar(exp_counts["exp_date"], exp_counts["count"], 
               color=colors, edgecolor="white", width=1.5)
        
        # Format x-axis to show dates nicely
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        ax.xaxis.set_major_locator(mdates.WeekdayLocator())
        plt.xticks(rotation=45, ha="right")
        
        ax.set_xlabel("Expiration Date")
        ax.set_ylabel("Number of Items")
        ax.set_title("Upcoming Expirations (Next 30 Days)")
        
        # Add a legend for the colors
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor="#e74c3c", label="≤3 days (urgent)"),
            Patch(facecolor="#f39c12", label="4-7 days (soon)"),
            Patch(facecolor="#27ae60", label="8+ days"),
        ]
        ax.legend(handles=legend_elements, loc="upper right")
        
        plt.tight_layout()
        plt.show()
    else:
        print("No items expiring in the next 30 days!")
else:
    print("No active inventory with expiration dates found.")

In [ ]:
# CHART: Purchase History Over Time
# Shows when shopping trips happened and how much was spent

if not receipts_df.empty and "purchase_date" in receipts_df.columns:
    receipts_df["date"] = pd.to_datetime(receipts_df["purchase_date"])
    
    # Group by purchase date: count items and total spending
    daily = receipts_df.groupby("date").agg(
        items=("item_name", "count"),
        total_spent=("total_price", "sum"),
    ).reset_index()
    
    fig, ax1 = plt.subplots(figsize=(10, 5))
    
    # Bar chart for number of items
    ax1.bar(daily["date"], daily["items"], color="#3498db", alpha=0.7,
            label="Items Purchased", width=1)
    ax1.set_xlabel("Date")
    ax1.set_ylabel("Number of Items", color="#3498db")
    ax1.tick_params(axis="y", labelcolor="#3498db")
    
    # Second y-axis for spending (if price data exists)
    if daily["total_spent"].notna().any():
        ax2 = ax1.twinx()  # Create a second y-axis sharing the same x-axis
        ax2.plot(daily["date"], daily["total_spent"], color="#e74c3c",
                 marker="o", linewidth=2, label="Total Spent ($)")
        ax2.set_ylabel("Total Spent ($)", color="#e74c3c")
        ax2.tick_params(axis="y", labelcolor="#e74c3c")
    
    ax1.set_title("Purchase History")
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No receipt data available to show purchase history.")

## 5. Chart Customization — Make It Yours

Every matplotlib element can be customized. Here are the most useful tweaks:

In [ ]:
# CUSTOMIZATION DEMO: The same data, three different looks

if not match_summary_df.empty:
    makeable = len(match_summary_df[match_summary_df["is_fully_makeable"] == True])
    close = len(match_summary_df[match_summary_df["missing_ingredients"].between(1, 2)])
    far = len(match_summary_df[match_summary_df["missing_ingredients"] >= 3])
    
    categories = ["Can Make\nNow", "Almost\n(1-2 missing)", "Need More\n(3+ missing)"]
    values = [makeable, close, far]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Style 1: Default
    axes[0].bar(categories, values, color=["#27ae60", "#f39c12", "#e74c3c"])
    axes[0].set_title("Default Style")
    
    # Style 2: Dark background
    axes[1].set_facecolor("#2c3e50")
    axes[1].bar(categories, values, color=["#1abc9c", "#f1c40f", "#e74c3c"],
                edgecolor="#2c3e50", linewidth=2)
    axes[1].set_title("Dark Style", color="#2c3e50")
    axes[1].tick_params(colors="#2c3e50")
    
    # Style 3: With annotations
    bars = axes[2].bar(categories, values, color=["#27ae60", "#f39c12", "#e74c3c"],
                       edgecolor="white", linewidth=2)
    for bar, val in zip(bars, values):
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                     str(val), ha="center", fontweight="bold", fontsize=14)
    axes[2].set_title("With Annotations")
    axes[2].set_ylim(0, max(values) * 1.3)  # Extra space for labels
    
    plt.suptitle("Same Data, Different Styles", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## Exercises

1. **Create a chart showing which recipes need the fewest additional ingredients.** Use a horizontal bar chart with `match_summary_df`, sorted by `missing_ingredients`.

2. **Make a stacked bar chart showing inventory by category AND source.** Hint: Use `pd.crosstab(inventory_df["category"], inventory_df["source"])` to get the data, then call `.plot(kind="barh", stacked=True)`.

3. **Create a chart of your own choosing** that answers a question about your data. Some ideas:
   - What's the average number of ingredients per weather category?
   - Which recipes have the shortest vs. longest cook times?
   - How does your inventory compare to what recipes need?

*Tip*: Start with the question you want to answer, then pick the chart type that best shows the answer.